In [1]:
from src.preferential_attachment_graph import PreferentialAttachmentGraph, FitnessType
from src.drawer import Drawer

In [2]:
G = PreferentialAttachmentGraph(n=50, m=2, fitness=FitnessType.POLY, fitness_alpha=1, track_history=False, select_planar_subgraph=True, test_planarity_restoring=True)
G.restoring_test_result[:5]

Planarity restoring test produced 1141 result files.


[{'added_edge': {'source': 0, 'target': 4},
  'min_edges_to_remove': 0,
  'removal_options': [[]]},
 {'added_edge': {'source': 0, 'target': 9},
  'min_edges_to_remove': 0,
  'removal_options': [[]]},
 {'added_edge': {'source': 0, 'target': 21},
  'min_edges_to_remove': 1,
  'removal_options': [[{'source': 10, 'target': 41}],
   [{'source': 18, 'target': 21}],
   [{'source': 18, 'target': 41}],
   [{'source': 4, 'target': 11}]]},
 {'added_edge': {'source': 2, 'target': 30},
  'min_edges_to_remove': 1,
  'removal_options': [[{'source': 18, 'target': 30}],
   [{'source': 4, 'target': 11}],
   [{'source': 0, 'target': 30}]]},
 {'added_edge': {'source': 32, 'target': 44},
  'min_edges_to_remove': 0,
  'removal_options': [[]]}]

In [ ]:
N_VALUES = [100, 500, 750]
M_VALUES = [2, 3]
FITNESS_FUNCTIONS = [
    {"type": FitnessType.LINEAR, "alpha": 1.0},
    {"type": FitnessType.POLY, "alpha": 0.2},
    {"type": FitnessType.POLY, "alpha": 0.5},
    {"type": FitnessType.POLY, "alpha": 0.8},
    {"type": FitnessType.LOG, "alpha": 1},
]
REPS = 10
RESULTS_DIR = "results/planarity_restoring"

In [ ]:
import os
import json
import itertools
import networkx as nx
from concurrent.futures import ProcessPoolExecutor, as_completed

import planarity_restoring_worker as worker

os.makedirs(RESULTS_DIR, exist_ok=True)


def get_result_filepath(params: dict) -> str:
    """Get the filepath for a given parameter combination."""
    filename = (
        f"n{params['n']}_m{params['m']}_{params['fitness_type'].name}_a{params['fitness_alpha']}_rep{params['rep']}_restoring.json"
    )
    return os.path.join(RESULTS_DIR, filename)


def is_already_done(params: dict) -> bool:
    """Check if a result file already exists and is valid."""
    filepath = get_result_filepath(params)
    if not os.path.exists(filepath):
        return False
    try:
        with open(filepath, "r") as f:
            data = json.load(f)
        return "restoring_results" in data and "summary" in data and "params" in data
    except (json.JSONDecodeError, IOError):
        return False


def generate_single(params: dict) -> dict:
    return worker.generate_single(params, RESULTS_DIR)

all_tasks = []
for n, m, fitness, rep in itertools.product(
    N_VALUES, M_VALUES, FITNESS_FUNCTIONS, range(REPS)
):
    all_tasks.append({
        "n": n,
        "m": m,
        "fitness_type": fitness["type"],
        "fitness_alpha": fitness["alpha"],
        "rep": rep,
    })

tasks = [t for t in all_tasks if not is_already_done(t)]
already_done = len(all_tasks) - len(tasks)

print(f"Total tasks: {len(all_tasks)} | Already done: {already_done} | Remaining: {len(tasks)}")

if not tasks:
    print("All planarity-restoring tasks already completed.")
else:
    all_results = []
    num_workers = 10

    print(f"Running with ProcessPoolExecutor ({num_workers} workers)...\n")
    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        futures = {executor.submit(worker.generate_single, task, RESULTS_DIR): task for task in tasks}
        for i, future in enumerate(as_completed(futures), 1):
            task = futures[future]
            try:
                result = future.result()
                all_results.append(result)
                summary = result["summary"]
                print(
                    f"[{i}/{len(tasks)}] n={task['n']} m={task['m']} {task['fitness_type'].name}(α={task['fitness_alpha']}) rep{task['rep']}: "
                    f"edges={summary['num_edges']}, restoring_cases={summary['num_restoring_cases']}"
                )
            except Exception as e:
                print(f"[{i}/{len(tasks)}] FAILED: {task} -> {e}")

    print(f"Done. Results saved to: {RESULTS_DIR}")

Total tasks: 500 | Already done: 0 | Remaining: 500
Running with ProcessPoolExecutor (10 workers)...

